# 02 - 多智能体辩论教程 (Multi-Agent Debate)

## 学习目标

1. **理解辩论系统** - 对抗性推理的原理和优势
2. **掌握辩论角色** - 正方、反方、裁判的职责
3. **实现辩论流程** - 开场、反驳、总结、评判
4. **应用场景** - 事实核查、决策支持、代码审查

---

## 理论背景

### 为什么需要辩论式 AI？

辩论式多智能体通过**对抗性推理**提高决策质量：

| 优势 | 描述 |
|------|------|
| 减少偏见 | 多角度审视问题 |
| 提高准确性 | 通过反驳发现漏洞 |
| 增强可解释性 | 论证过程透明 |

In [ ]:
# 环境设置
import sys
import asyncio
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from agent_base import AgentConfig, AgentRole, MockLLM
from debate_agents import (
    DebateRole, DebateConfig, Argument,
    DebaterAgent, JudgeAgent, DebateArena
)

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    return loop.run_until_complete(coro)

print('=' * 60)
print('多智能体系统 - 辩论教程')
print('=' * 60)

---

## 第一部分：辩论角色

| 角色 | 描述 | 职责 |
|------|------|------|
| PROPONENT | 正方 | 支持论题 |
| OPPONENT | 反方 | 反对论题 |
| JUDGE | 裁判 | 评估裁决 |

In [ ]:
# 1.1 查看辩论角色
print('辩论角色:')
for role in DebateRole:
    print(f'  {role.name}: {role.value}')

---

## 第二部分：创建辩论者

In [ ]:
# 2.1 创建正方辩手
print('=' * 60)
print('创建辩手')
print('=' * 60)

pro_llm = MockLLM(responses=[
    'AI提高生产效率，医疗突破，科学研究加速。',
    '历史证明技术革命创造更多就业机会。',
    '总结：AI益处远大于风险。'
])
pro_config = AgentConfig(name='正方', role=AgentRole.DEBATER)
proponent = DebaterAgent(pro_config, DebateRole.PROPONENT, pro_llm)
print(f'正方: {proponent.name}')

# 创建反方辩手
opp_llm = MockLLM(responses=[
    'AI导致失业，隐私侵犯，安全风险。',
    '技术革命速度前所未有，社会难以适应。',
    '总结：AI风险不容忽视。'
])
opp_config = AgentConfig(name='反方', role=AgentRole.DEBATER)
opponent = DebaterAgent(opp_config, DebateRole.OPPONENT, opp_llm)
print(f'反方: {opponent.name}')

---

## 第三部分：创建裁判

In [ ]:
# 3.1 创建裁判
judge_llm = MockLLM(responses=[
    '正方得分: 8/10, 反方得分: 7/10。正方获胜。'
])
judge_config = AgentConfig(name='裁判', role=AgentRole.CRITIC)
judge = JudgeAgent(judge_config, llm=judge_llm)
print(f'裁判: {judge.name}')
print(f'评判标准: {judge.criteria}')

---

## 第四部分：配置和运行辩论

In [ ]:
# 4.1 配置辩论
config = DebateConfig(
    topic='AI对人类社会利大于弊',
    max_rounds=2,
    allow_rebuttals=True
)
print(f'辩题: {config.topic}')
print(f'轮数: {config.max_rounds}')

In [ ]:
# 4.2 创建辩论场并运行
arena = DebateArena(proponent, opponent, judge, config)

async def run_debate():
    result = await arena.run_debate()
    print(f'\n获胜方: {result.winner_role}')
    print(f'正方得分: {result.proponent_score.total}')
    print(f'反方得分: {result.opponent_score.total}')
    return result

result = run_async(run_debate())

---

## 第五部分：手动辩论流程

In [ ]:
# 5.1 手动生成开场陈述
pro2_llm = MockLLM(responses=['AI提高医疗诊断准确率。'])
opp2_llm = MockLLM(responses=['AI诊断错误责任难界定。'])

pro2 = DebaterAgent(AgentConfig(name='正方2'), DebateRole.PROPONENT, pro2_llm)
opp2 = DebaterAgent(AgentConfig(name='反方2'), DebateRole.OPPONENT, opp2_llm)

async def manual_debate():
    topic = 'AI应用于医疗诊断'
    print(f'辩题: {topic}')
    
    pro_arg = await pro2.generate_opening(topic)
    print(f'正方: {pro_arg.content}')
    
    opp_arg = await opp2.generate_opening(topic)
    print(f'反方: {opp_arg.content}')

run_async(manual_debate())

---

## 总结

本教程涵盖：
1. **DebateRole**: 正方、反方、裁判
2. **DebaterAgent**: 辩论者创建
3. **JudgeAgent**: 裁判评判
4. **DebateArena**: 自动化辩论

### 下一步
学习 **03_CollaborativeAgents_tutorial.ipynb** 了解协作式多智能体